# Evaluating LLM Checkpoints with NeMo AutoModel

This tutorial demonstrates how to evaluate LLM checkpoints produced by
[NeMo AutoModel](https://github.com/NVIDIA-NeMo/Automodel) training.

After training a model -- whether through Domain Adaptive Pre-Training (DAPT),
Supervised Fine-Tuning (SFT), or Parameter-Efficient Fine-Tuning (PEFT) --
evaluation measures how well the model performs on tasks it was never explicitly
trained on. This is the standard way to assess whether training improved
(or degraded) the model's capabilities.

We cover two complementary approaches:

1. **Standardized benchmarks** using
   [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
   with a [vLLM](https://github.com/vllm-project/vllm) backend -- the
   recommended approach for reproducible, publishable results.
2. **Custom evaluation** using HuggingFace `transformers` for quick,
   interactive spot-checks during development.

### Benchmarks at a glance

| Benchmark | What it measures | Typical use |
|-----------|-----------------|-------------|
| MMLU | World knowledge, reasoning | General capability |
| HellaSwag | Commonsense reasoning | General capability |
| GSM8K | Math reasoning | Reasoning models |
| IFEval | Instruction following | SFT / chat models |
| Winogrande | Coreference resolution | General capability |

### Prerequisites

* NVIDIA GPU(s) with enough memory for the model being evaluated.
* A trained checkpoint **or** a HuggingFace model name for demonstration.

## Step 0 -- Environment Setup

Verify that PyTorch can see a CUDA device.

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1 -- Locate the Checkpoint

NeMo AutoModel saves checkpoints with the following layout:

```
checkpoints/
  epoch_0_step_200/
    model/
      consolidated/          <-- HuggingFace-compatible checkpoint
        config.json
        model-00001-of-00002.safetensors
        model.safetensors.index.json
        tokenizer.json
        ...
  epoch_0_step_400/
    ...
```

The `model/consolidated/` directory inside each checkpoint is a standard
HuggingFace model directory. Both `lm-evaluation-harness` and `transformers`
can load it directly.

For **PEFT** checkpoints, adapter weights are saved separately and must be
merged before evaluation (or loaded with the appropriate adapter library).

Update `CKPT_BASE` below to point at your checkpoint directory.

In [ ]:
import glob
import os

# -- Update this path to your actual checkpoint directory --
CKPT_BASE = "checkpoints/"

ckpt_dirs = sorted(glob.glob(os.path.join(CKPT_BASE, "epoch_*_step_*")))
if ckpt_dirs:
    latest = ckpt_dirs[-1]
    consolidated = os.path.join(latest, "model", "consolidated")
    print(f"Latest checkpoint: {latest}")
    print(f"Consolidated model: {consolidated}")
    if os.path.isdir(consolidated):
        print(f"Files: {os.listdir(consolidated)[:10]}")
    else:
        print(f"WARNING: {consolidated} does not exist -- check checkpoint layout")
else:
    print(f"No checkpoints found in {CKPT_BASE}")
    print("Using a HuggingFace model for demo: meta-llama/Llama-3.2-1B")
    consolidated = "meta-llama/Llama-3.2-1B"

print(f"\nModel path for evaluation: {consolidated}")

## Step 2 -- Install lm-evaluation-harness

[lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
is the standard open-source framework for evaluating language models. Installing
it with the `vllm` extra gives us a high-throughput inference backend that
supports continuous batching, paged attention, and tensor parallelism.

In [ ]:
!pip install lm-eval[vllm] 2>&1 | tail -5

Verify the installation:

In [ ]:
!lm_eval --help 2>&1 | head -15

## Step 3 -- Run Standard Benchmarks

The `lm_eval` CLI is the simplest way to evaluate a model. Key options:

| Flag | Purpose |
|------|---------|
| `--model vllm` | Use the vLLM backend for fast inference |
| `--model_args pretrained=<path>` | Path to HF model or local checkpoint |
| `--tasks <task>` | Comma-separated list of benchmarks |
| `--batch_size auto` | Let vLLM choose the optimal batch size |
| `--output_path <dir>` | Directory to save results JSON |

The commands below use the `consolidated` variable set in Step 1.

### HellaSwag (commonsense reasoning, fast)

HellaSwag is a good first benchmark to run: it is relatively quick (~10k
samples) and reliably distinguishes between model quality levels.

In [ ]:
# Evaluate on HellaSwag
!lm_eval --model vllm \
    --model_args pretrained={consolidated},dtype=auto,gpu_memory_utilization=0.8 \
    --tasks hellaswag \
    --batch_size auto \
    --output_path results/hellaswag

### MMLU (world knowledge and reasoning, comprehensive)

MMLU tests knowledge across 57 subjects. It takes longer to run but gives a
broad picture of the model's capabilities.

In [ ]:
# Evaluate on MMLU
!lm_eval --model vllm \
    --model_args pretrained={consolidated},dtype=auto,gpu_memory_utilization=0.8 \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size auto \
    --output_path results/mmlu

## Step 4 -- Parse and Compare Results

`lm_eval` writes a `results.json` file inside the output directory. The
helper function below reads it and extracts the numeric metrics.

A typical workflow is to evaluate the **base model** and the **fine-tuned
model** on the same benchmarks and compare the numbers side by side.

In [ ]:
import json
from pathlib import Path


def load_results(result_dir):
    """Load lm-eval results from an output directory.

    Returns a flat dictionary of ``task/metric -> value`` pairs.
    """
    result_files = list(Path(result_dir).rglob("results.json"))
    if not result_files:
        print(f"No results found in {result_dir}")
        return {}

    with open(result_files[0]) as f:
        data = json.load(f)

    results = {}
    for task, metrics in data.get("results", {}).items():
        for metric, value in metrics.items():
            if isinstance(value, (int, float)):
                results[f"{task}/{metric}"] = value
    return results


def compare_results(base_dir, finetuned_dir):
    """Print a side-by-side comparison of base vs fine-tuned results."""
    base = load_results(base_dir)
    ft = load_results(finetuned_dir)

    all_keys = sorted(set(base) | set(ft))
    print(f"{'Metric':<45} {'Base':>10} {'Fine-tuned':>12} {'Delta':>10}")
    print("-" * 80)
    for key in all_keys:
        b = base.get(key)
        f = ft.get(key)
        b_str = f"{b:.4f}" if b is not None else "--"
        f_str = f"{f:.4f}" if f is not None else "--"
        if b is not None and f is not None:
            delta = f - b
            d_str = f"{delta:+.4f}"
        else:
            d_str = "--"
        print(f"{key:<45} {b_str:>10} {f_str:>12} {d_str:>10}")


# Example usage (uncomment after running evaluations for both models):
# compare_results("results/hellaswag_base", "results/hellaswag_finetuned")

In [ ]:
# Inspect results from a single evaluation run:
results = load_results("results/hellaswag")
for metric, value in sorted(results.items()):
    print(f"  {metric}: {value:.4f}")

## Step 5 -- Custom Evaluation: Quick Spot-Check

For rapid iteration during development, it is sometimes easier to load the
model with `transformers` and run a few hand-picked prompts. This is **not**
a substitute for proper benchmarks, but it can catch obvious regressions or
confirm that the model has learned the desired behavior.

Because AutoModel saves checkpoints in HuggingFace format, they load
directly with `AutoModelForCausalLM`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "meta-llama/Llama-3.2-1B"  # Replace with your checkpoint path

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print(f"Model loaded: {model_path}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test prompts -- adjust these to match your training domain.
prompts = [
    "The capital of France is",
    "def fibonacci(n):\n",
    "Explain quantum computing in simple terms:",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
        )
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Prompt:   {prompt}")
    print(f"Response: {response}")
    print("-" * 60)

## Step 6 -- Evaluating Instruction-Tuned Models with Chat Template

After SFT, models expect inputs wrapped in a chat template. The
`--apply_chat_template` flag tells `lm_eval` to format prompts using the
tokenizer's built-in chat template.

**IFEval** (Instruction Following Evaluation) is specifically designed to
measure how well a model follows explicit instructions (e.g., "write
exactly 3 paragraphs", "include the word 'ocean'"). It is the recommended
benchmark for SFT and PEFT models.

> **Note:** The `--apply_chat_template` flag requires a model whose tokenizer
> includes a chat template. Base models such as `meta-llama/Llama-3.2-1B` do
> not ship with one. Point `consolidated` at an instruct/chat model
> (e.g., `meta-llama/Llama-3.2-1B-Instruct`) or your own SFT checkpoint
> before running this cell.

In [ ]:
# Evaluate instruction following -- use an instruct/chat model for this step.
!lm_eval --model vllm \
    --model_args pretrained={consolidated},dtype=auto,gpu_memory_utilization=0.8 \
    --tasks ifeval \
    --batch_size auto \
    --apply_chat_template \
    --fewshot_as_multiturn \
    --output_path results/ifeval

## Step 7 -- Multi-GPU Evaluation

Larger models may not fit on a single GPU. vLLM supports tensor parallelism
natively -- set `tensor_parallel_size` in the model args to split the model
across multiple GPUs.

For example, to evaluate on 2 GPUs:

In [ ]:
# Multi-GPU evaluation with tensor parallelism.
# Adjust tensor_parallel_size to match your GPU count.
!lm_eval --model vllm \
    --model_args pretrained={consolidated},dtype=auto,tensor_parallel_size=2,gpu_memory_utilization=0.8 \
    --tasks mmlu \
    --batch_size auto \
    --output_path results/mmlu_tp2

## Summary

### Recommended evaluation strategy by training stage

| Training stage | Recommended benchmarks | Notes |
|----------------|----------------------|-------|
| **DAPT** | Domain-specific perplexity; MMLU, HellaSwag | Check for no degradation on general benchmarks |
| **SFT** | IFEval; task-specific metrics (EM, F1 for QA) | Use `--apply_chat_template` |
| **PEFT** | Same as SFT | Merge adapters or load with adapter support |
| **Reasoning SFT** | GSM8K, MATH; domain reasoning tasks | Evaluate both with and without chain-of-thought |

### Quick reference: lm_eval commands

```bash
# Base model, single GPU
lm_eval --model vllm \
    --model_args pretrained=<path>,dtype=auto,gpu_memory_utilization=0.8 \
    --tasks hellaswag,mmlu \
    --batch_size auto \
    --output_path results/

# SFT model with chat template
lm_eval --model vllm \
    --model_args pretrained=<path>,dtype=auto \
    --tasks ifeval \
    --batch_size auto \
    --apply_chat_template \
    --fewshot_as_multiturn \
    --output_path results/

# Large model, multi-GPU
lm_eval --model vllm \
    --model_args pretrained=<path>,dtype=auto,tensor_parallel_size=4 \
    --tasks mmlu \
    --batch_size auto \
    --output_path results/
```

## Resources

* [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) -- benchmark framework
* [vLLM](https://github.com/vllm-project/vllm) -- high-throughput inference engine
* [NeMo AutoModel](https://github.com/NVIDIA-NeMo/Automodel) -- training framework
* [Open LLM Leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard) -- community benchmark results
* [DAPT Tutorial](../dapt/) -- domain adaptive pre-training with NeMo AutoModel
* [SFT / PEFT Tutorial](../sft-peft/) -- supervised and parameter-efficient fine-tuning
* [Reasoning SFT Tutorial](../reasoning-sft/) -- reasoning-focused supervised fine-tuning